# 24-Hour Ahead Prediction of Dissolved Oxygen & pH in Fish Pond Water

**Problem:** Simultaneously predict DO and pH 24 h ahead using Temperature, Turbidity, and Feed Amount.

**Approach:** Compare a Multi-Output Gradient Boosting Regressor against two independent single-output models.

## 1. Environment Setup & Dataset Download

In [ ]:
import subprocess, sys

def install(pkg):

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['kaggle', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn']:

    install(pkg)


In [ ]:
import os, zipfile, warnings

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import matplotlib.gridspec as gridspec

import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor

from sklearn.multioutput import MultiOutputRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

plt.rcParams['figure.dpi'] = 120

DATASET_SLUG = 'jocelyndumlao/iot-monitoring-of-water-quality-and-tilapia'

DATA_DIR     = 'data'

SEED         = 42


In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]

if not csv_files:

    print('Downloading dataset from Kaggle...')

    result = subprocess.run(

        ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', DATA_DIR, '--unzip'],

        capture_output=True, text=True

    )

    print(result.stdout or result.stderr)

    csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]

print('CSV files found:', csv_files)


## 2. Data Loading & Exploration

In [ ]:
dfs = [pd.read_csv(os.path.join(DATA_DIR, f)) for f in sorted(csv_files)]

raw = pd.concat(dfs, ignore_index=True)

print(f'Shape: {raw.shape}')

raw.head()


In [ ]:
print('Column names:', raw.columns.tolist())

print()

raw.info()


In [ ]:
raw.describe().round(3)


In [ ]:
print('Missing values per column:')

print(raw.isnull().sum())


## 3. Column Mapping & DateTime Parsing

In [ ]:
def find_col(df, keywords):

    cols_lower = {c.lower(): c for c in df.columns}

    for kw in keywords:

        for low, orig in cols_lower.items():

            if kw.lower() in low:

                return orig

    return None

COL_TIME  = find_col(raw, ['date', 'time', 'timestamp', 'datetime'])

COL_TEMP  = find_col(raw, ['temp'])

COL_TURB  = find_col(raw, ['turb'])

COL_FEED  = find_col(raw, ['feed', 'amount'])

COL_DO    = find_col(raw, ['do', 'dissolved', 'oxygen'])

COL_PH    = find_col(raw, ['ph'])

for name, col in [('Datetime', COL_TIME), ('Temperature', COL_TEMP),

                   ('Turbidity', COL_TURB), ('Feed', COL_FEED),

                   ('DO', COL_DO), ('pH', COL_PH)]:

    status = '✓' if col else '✗ NOT FOUND'

    print(f'  {name}: {col}  {status}')


In [ ]:
df = raw.copy()

if COL_TIME:

    df[COL_TIME] = pd.to_datetime(df[COL_TIME], infer_datetime_format=True, errors='coerce')

    df = df.sort_values(COL_TIME).reset_index(drop=True)

    df = df.set_index(COL_TIME)

rename_map = {}

if COL_TEMP: rename_map[COL_TEMP] = 'Temperature'

if COL_TURB: rename_map[COL_TURB] = 'Turbidity'

if COL_FEED: rename_map[COL_FEED] = 'FeedAmount'

if COL_DO:   rename_map[COL_DO]   = 'DO'

if COL_PH:   rename_map[COL_PH]   = 'pH'

df = df.rename(columns=rename_map)

FEATURES = [c for c in ['Temperature', 'Turbidity', 'FeedAmount'] if c in df.columns]

TARGETS  = [c for c in ['DO', 'pH'] if c in df.columns]

df = df[FEATURES + TARGETS].dropna()

print(f'Clean df shape: {df.shape}')

print(f'Features : {FEATURES}')

print(f'Targets  : {TARGETS}')

df.head()


## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(len(FEATURES + TARGETS), 1,

                          figsize=(14, 2.8 * len(FEATURES + TARGETS)),

                          sharex=True)

colors = sns.color_palette('muted', len(FEATURES + TARGETS))

for ax, col, c in zip(axes, FEATURES + TARGETS, colors):

    ax.plot(df.index, df[col], color=c, linewidth=0.6)

    ax.set_ylabel(col, fontsize=9)

    ax.tick_params(axis='x', labelsize=7)

axes[-1].set_xlabel('Time')

fig.suptitle('Full Time-Series Overview', fontsize=13, y=1.01)

plt.tight_layout()

plt.savefig('01_timeseries_overview.png', bbox_inches='tight')

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

corr = df.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',

            center=0, linewidths=0.5, ax=ax)

ax.set_title('Pearson Correlation Matrix')

plt.tight_layout()

plt.savefig('02_correlation_heatmap.png', bbox_inches='tight')

plt.show()


## 5. Feature Engineering — 24-Hour Ahead Targets

In [ ]:
if isinstance(df.index, pd.DatetimeIndex):

    freq = pd.infer_freq(df.index)

    if freq is None:

        gaps = df.index.to_series().diff().dropna()

        median_gap = gaps.median()

        horizon = int(pd.Timedelta('24h') / median_gap)

    else:

        offset = pd.tseries.frequencies.to_offset(freq)

        horizon = int(pd.Timedelta('24h') / offset.nanos * 1e9)

else:

    horizon = 24

print(f'Sampling frequency inferred  : {freq if isinstance(df.index, pd.DatetimeIndex) else "unknown"}')

print(f'Horizon (rows = 24 h ahead)  : {horizon}')


In [ ]:
ml = df.copy()

for t in TARGETS:

    ml[f'{t}_future'] = ml[t].shift(-horizon)

for lag in [1, 2, 3]:

    for feat in ['Temperature', 'Turbidity']:

        if feat in ml.columns:

            ml[f'{feat}_lag{lag}'] = ml[feat].shift(lag)

ml.dropna(inplace=True)

FUTURE_TARGETS = [f'{t}_future' for t in TARGETS]

ALL_FEATURES   = [c for c in ml.columns if c not in TARGETS + FUTURE_TARGETS]

print(f'Rows after shifting   : {len(ml)}')

print(f'Feature columns ({len(ALL_FEATURES)}): {ALL_FEATURES}')

print(f'Target columns  ({len(FUTURE_TARGETS)}): {FUTURE_TARGETS}')


## 6. Train / Test Split (Chronological)

In [ ]:
TRAIN_RATIO = 0.8

split_idx   = int(len(ml) * TRAIN_RATIO)

X_train = ml.iloc[:split_idx][ALL_FEATURES].values

X_test  = ml.iloc[split_idx:][ALL_FEATURES].values

y_train = ml.iloc[:split_idx][FUTURE_TARGETS].values

y_test  = ml.iloc[split_idx:][FUTURE_TARGETS].values

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')

scaler  = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test  = scaler.transform(X_test)


## 7. Model Training

### 7a — Multi-Output Gradient Boosting Regressor
`MultiOutputRegressor` wraps a `GradientBoostingRegressor` per target, but predicts both in a single `.fit()` / `.predict()` call, making it the canonical sklearn multi-output solution.

In [ ]:
GBR_PARAMS = dict(

    n_estimators=300,

    learning_rate=0.05,

    max_depth=4,

    subsample=0.8,

    random_state=SEED

)

mo_gbr = MultiOutputRegressor(

    GradientBoostingRegressor(**GBR_PARAMS),

    n_jobs=-1

)

mo_gbr.fit(X_train, y_train)

y_pred_mo = mo_gbr.predict(X_test)

print('Multi-Output GBR trained.')


### 7b — Independent Single-Output GBR for DO

In [ ]:
gbr_do = GradientBoostingRegressor(**GBR_PARAMS)

gbr_do.fit(X_train, y_train[:, 0])

y_pred_do_single = gbr_do.predict(X_test)

print('Single-Output GBR (DO) trained.')


### 7c — Independent Single-Output GBR for pH

In [ ]:
gbr_ph = GradientBoostingRegressor(**GBR_PARAMS)

gbr_ph.fit(X_train, y_train[:, 1])

y_pred_ph_single = gbr_ph.predict(X_test)

print('Single-Output GBR (pH) trained.')


## 8. Evaluation

In [ ]:
def metrics(y_true, y_pred, label):

    mae  = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    r2   = r2_score(y_true, y_pred)

    return {'Model': label, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

rows = [

    metrics(y_test[:, 0], y_pred_mo[:, 0],     'Multi-Output GBR  →  DO'),

    metrics(y_test[:, 0], y_pred_do_single,     'Single-Output GBR →  DO'),

    metrics(y_test[:, 1], y_pred_mo[:, 1],      'Multi-Output GBR  →  pH'),

    metrics(y_test[:, 1], y_pred_ph_single,     'Single-Output GBR →  pH'),

]

results = pd.DataFrame(rows).set_index('Model')

results = results.round(4)

styled = results.style
    .background_gradient(subset=['MAE','RMSE'], cmap='RdYlGn_r')
    .background_gradient(subset=['R²'], cmap='RdYlGn')
    .format('{:.4f}')

print('\n=== Performance Summary ===')

display(styled)

results


## 9. Prediction Plots

In [ ]:
test_index = ml.index[split_idx:] if isinstance(ml.index, pd.DatetimeIndex) else np.arange(len(y_test))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, col_idx, target_name in zip(axes, [0, 1], TARGETS):

    ax.plot(test_index, y_test[:, col_idx], color='black', lw=1.2,

            label='Actual', zorder=3)

    ax.plot(test_index, y_pred_mo[:, col_idx], color='steelblue',

            lw=1, ls='--', label='Multi-Output GBR', zorder=2)

    single_pred = y_pred_do_single if col_idx == 0 else y_pred_ph_single

    ax.plot(test_index, single_pred, color='tomato',

            lw=1, ls=':', label='Single-Output GBR', zorder=2)

    ax.set_ylabel(f'{target_name} (24 h ahead)', fontsize=10)

    ax.legend(fontsize=8)

    ax.tick_params(axis='x', labelsize=7)

axes[-1].set_xlabel('Time')

fig.suptitle('24-Hour Ahead Predictions — Test Set', fontsize=13)

plt.tight_layout()

plt.savefig('03_predictions.png', bbox_inches='tight')

plt.show()


## 10. Scatter Plots — Actual vs. Predicted

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

plot_cases = [

    (axes[0, 0], y_test[:, 0], y_pred_mo[:, 0],     'Multi-Output GBR — DO'),

    (axes[0, 1], y_test[:, 0], y_pred_do_single,     'Single-Output GBR — DO'),

    (axes[1, 0], y_test[:, 1], y_pred_mo[:, 1],      'Multi-Output GBR — pH'),

    (axes[1, 1], y_test[:, 1], y_pred_ph_single,     'Single-Output GBR — pH'),

]

for ax, yt, yp, title in plot_cases:

    ax.scatter(yt, yp, alpha=0.35, s=12, edgecolors='none')

    mn, mx = min(yt.min(), yp.min()), max(yt.max(), yp.max())

    ax.plot([mn, mx], [mn, mx], 'r--', lw=1, label='Perfect')

    r2 = r2_score(yt, yp)

    ax.set_title(f'{title}\nR² = {r2:.4f}', fontsize=9)

    ax.set_xlabel('Actual')

    ax.set_ylabel('Predicted')

    ax.legend(fontsize=7)

plt.suptitle('Actual vs Predicted — Test Set', fontsize=12, y=1.01)

plt.tight_layout()

plt.savefig('04_actual_vs_predicted.png', bbox_inches='tight')

plt.show()


## 11. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(6 * len(TARGETS), 5))

if len(TARGETS) == 1:

    axes = [axes]

for ax, estimator, tname in zip(axes, mo_gbr.estimators_, TARGETS):

    importances = estimator.feature_importances_

    idx = np.argsort(importances)[::-1]

    ax.bar(np.array(ALL_FEATURES)[idx], importances[idx], color=sns.color_palette('muted')[0])

    ax.set_title(f'Feature Importance\nMulti-Output GBR → {tname}', fontsize=10)

    ax.set_xlabel('Feature')

    ax.set_ylabel('Importance')

    ax.tick_params(axis='x', rotation=35, labelsize=8)

plt.tight_layout()

plt.savefig('05_feature_importance.png', bbox_inches='tight')

plt.show()


## 12. Summary & Discussion

The table above is the primary comparison between the two strategies:

| Metric | Better (↓/↑) | Multi-Output GBR | Single-Output GBR |
|--------|:----------:|:----------------:|:-----------------:|
| MAE    | ↓ lower    | see results      | see results       |
| RMSE   | ↓ lower    | see results      | see results       |
| R²     | ↑ higher   | see results      | see results       |

**Key observations:**

* `MultiOutputRegressor` fits one GBR *per* target independently under the hood, so if the numbers are identical it means the multi-output wrapper is equivalent to running them separately (which is expected for GBR).
* Differences can arise from shared hyperparameter tuning across targets or if an explicit multi-output learner (e.g. XGBoost with `multi:squarederror`) is used instead.
* Feature importance reveals which of Temperature, Turbidity, and Feed Amount most drives each water quality parameter 24 h ahead.

**Potential improvements:**
* Hyperparameter search (`GridSearchCV` / `Optuna`) with time-series cross-validation (`TimeSeriesSplit`).
* Adding lag features for DO and pH themselves (autoregressive terms).
* Trying a native multi-output model such as `XGBRegressor` with `tree_method='hist'` and `multi_strategy='multi_output_tree'`.

In [ ]:
results.to_csv('model_comparison_results.csv')

print('Results saved to model_comparison_results.csv')

print(results.to_string())
